# 01 — Исследование данных (EDA)

Цель: понять распределение классов, качество изображений, статистику датасета DENTEX.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

META = Path('../data/processed/metadata.csv')
IMG_DIR = Path('../data/processed/images')
MASK_DIR = Path('../data/processed/masks')

In [ ]:
df = pd.read_csv(META)
print(f'Total images: {len(df)}')
df.head()

In [ ]:
# Class distribution in masks
class_names = ['background', 'tooth', 'bone', 'cortical']
pixel_counts = np.zeros(4, dtype=np.int64)

for mask_path in list(MASK_DIR.glob('*.png'))[:200]:  # sample 200
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    for c in range(4):
        pixel_counts[c] += (m == c).sum()

fracs = pixel_counts / pixel_counts.sum()
plt.figure(figsize=(8, 4))
plt.bar(class_names, fracs * 100)
plt.ylabel('% pixels')
plt.title('Class distribution in masks')
plt.tight_layout()
plt.show()

print('Class weights (inverse frequency):')
inv = 1.0 / (fracs + 1e-6)
inv_norm = inv / inv.sum()
for name, w in zip(class_names, inv_norm):
    print(f'  {name}: {w:.4f}')

In [ ]:
# Sample images + masks
samples = list(IMG_DIR.glob('*.png'))[:6]
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, img_path in enumerate(samples):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(MASK_DIR / img_path.name), cv2.IMREAD_GRAYSCALE)
    axes[0, i].imshow(img, cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(mask, cmap='tab10', vmin=0, vmax=3); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Image'); axes[1, 0].set_ylabel('Mask')
plt.suptitle('OPG samples + segmentation masks')
plt.tight_layout()
plt.show()

In [ ]:
# Image intensity statistics
means, stds = [], []
for p in list(IMG_DIR.glob('*.png'))[:100]:
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(float) / 255
    means.append(img.mean())
    stds.append(img.std())

print(f'Mean brightness: {np.mean(means):.3f} ± {np.std(means):.3f}')
print(f'Mean contrast:   {np.mean(stds):.3f} ± {np.std(stds):.3f}')